In [62]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score

In [38]:
matches = pd.read_csv('matches.csv')

In [39]:
matches

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,notes,sh,sot,dist,pk,pkatt,season,team,xg,xga
0,2023-07-22,15:15,Premier League,Matchweek 1,Sat,Away,W,2,0,Nizhny Novgorod,...,NaN,11.0,2.0,NaN,0,0,2023,Zenit,NaN,NaN
1,2023-07-29,20:00,Premier League,Matchweek 2,Sat,Away,D,1,1,Rostov,...,NaN,6.0,1.0,NaN,0,0,2023,Zenit,NaN,NaN
2,2023-08-06,19:00,Premier League,Matchweek 3,Sun,Home,L,2,3,Dynamo Mosc,...,NaN,15.0,5.0,NaN,0,0,2023,Zenit,NaN,NaN
3,2023-08-13,17:30,Premier League,Matchweek 4,Sun,Home,W,2,0,Fakel Voronezh,...,NaN,10.0,6.0,NaN,0,0,2023,Zenit,NaN,NaN
4,2023-08-20,19:30,Premier League,Matchweek 5,Sun,Away,W,3,1,Spartak Moscow,...,NaN,12.0,3.0,NaN,0,0,2023,Zenit,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2383,2020-07-05,18:30,Premier League,Matchweek 26,Sun,Away,L,0,1,Rubin Kazan,...,NaN,15.0,3.0,NaN,0,0,2019,Orenburg,NaN,NaN
2384,2020-07-08,20:00 (18:00),Premier League,Matchweek 27,Wed,Home,L,0,4,CSKA Moscow,...,NaN,9.0,3.0,NaN,0,0,2019,Orenburg,NaN,NaN
2385,2020-07-12,20:30 (18:30),Premier League,Matchweek 28,Sun,Home,D,0,0,Rostov,...,NaN,8.0,3.0,NaN,0,0,2019,Orenburg,NaN,NaN
2386,2020-07-15,20:30,Premier League,Matchweek 29,Wed,Away,L,1,4,Zenit,...,NaN,16.0,3.0,NaN,0,0,2019,Orenburg,NaN,NaN


In [40]:
matches.shape

(2388, 27)

In [41]:
matches['team'].value_counts()

team
Zenit                     150
Dynamo Moscow             150
Spartak Moscow            150
Lokomotiv Moscow          150
CSKA Moscow               150
Akhmat Grozny             150
Ural Yekaterinburg        149
Rostov                    149
Krasnodar                 148
Sochi                     148
Rubin Kazan               120
Samara                    119
FC Khimki                  90
Nizhny Novgorod            90
Arsenal Tula               90
Ufa                        90
Orenburg                   88
Fakel Voronezh             60
Tambov                     59
FC Baltika Kaliningrad     30
Torpedo Moscow             30
Rotor Volgograd            28
Name: count, dtype: int64

In [42]:
matches['round'].value_counts()

round
Matchweek 1     80
Matchweek 2     80
Matchweek 3     80
Matchweek 4     80
Matchweek 5     80
Matchweek 6     80
Matchweek 10    80
Matchweek 9     80
Matchweek 12    80
Matchweek 11    80
Matchweek 13    80
Matchweek 14    80
Matchweek 16    80
Matchweek 15    80
Matchweek 26    80
Matchweek 21    80
Matchweek 17    80
Matchweek 18    80
Matchweek 19    80
Matchweek 20    80
Matchweek 22    80
Matchweek 23    80
Matchweek 28    80
Matchweek 27    80
Matchweek 8     78
Matchweek 7     78
Matchweek 24    78
Matchweek 25    78
Matchweek 29    78
Matchweek 30    78
Name: count, dtype: int64

In [43]:
matches['date'] = pd.to_datetime(matches['date'])

In [44]:
matches.dtypes

date             datetime64[ns]
time                     object
comp                     object
round                    object
day                      object
venue                    object
result                   object
gf                        int64
ga                        int64
opponent                 object
poss                    float64
attendance              float64
captain                  object
formation                object
opp formation            object
referee                  object
match report             object
notes                    object
sh                      float64
sot                     float64
dist                    float64
pk                        int64
pkatt                     int64
season                    int64
team                     object
xg                      float64
xga                     float64
dtype: object

In [45]:
matches.isna().sum()

date                0
time                0
comp                0
round               0
day                 0
venue               0
result              0
gf                  0
ga                  0
opponent            0
poss               14
attendance        116
captain             2
formation           0
opp formation       0
referee             0
match report        0
notes            2386
sh                  0
sot                 0
dist             2388
pk                  0
pkatt               0
season              0
team                0
xg               2388
xga              2388
dtype: int64

In [46]:
matches['venue_code'] = matches['venue'].astype('category').cat.codes

In [47]:
matches['opp_code'] = matches['opponent'].astype('category').cat.codes

In [48]:
matches['hour'] = matches['time'].str.split(':').str[0].astype('int')

In [49]:
matches['day_code'] = matches['date'].dt.dayofweek

In [50]:
matches["target"] = (matches["result"] == "W").astype("int")

In [51]:
RFC = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1) 

In [52]:
df_train = matches[matches['date'] < '2023-08-01']
df_test = matches[matches['date'] >= '2023-08-01'] 

In [53]:
predictors = ['venue_code', 'opp_code', 'hour', 'day_code']

In [54]:
RFC.fit(df_train[predictors], df_train['target'])

,n_estimators,50
,criterion,'gini'
,max_depth,None
,min_samples_split,10
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [55]:
pred = RFC.predict(df_test[predictors])

In [56]:
acc = accuracy_score(df_test['target'], pred)

In [57]:
acc 

0.5758928571428571

In [60]:
combined = pd.DataFrame(dict(actual=df_test["target"], predicted=pred))

In [61]:
pd.crosstab(index=combined["actual"], columns=combined["predicted"])

predicted,0,1
actual,,
0,220,65
1,125,38


In [63]:
prec = precision_score(df_test["target"], pred)
print(prec)

0.36893203883495146


In [64]:
grouped_matches = matches.groupby('team')

In [ ]:
group = grouped_matches.get_group('Zenit').sort_values("date")

In [66]:
group

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,pkatt,season,team,xg,xga,venue_code,opp_code,hour,day_code,target
1916,2019-07-14,19:00,Premier League,Matchweek 1,Sun,Home,W,2,1,Tambov,...,0,2019,Zenit,NaN,NaN,1,17,19,6,1
1917,2019-07-21,21:30,Premier League,Matchweek 2,Sun,Away,W,2,0,Sochi,...,1,2019,Zenit,NaN,NaN,0,15,21,6,1
1918,2019-07-28,16:00 (14:00),Premier League,Matchweek 3,Sun,Away,W,2,0,Orenburg,...,1,2019,Zenit,NaN,NaN,0,10,16,6,1
1919,2019-08-03,21:30,Premier League,Matchweek 4,Sat,Home,D,1,1,Krasnodar,...,0,2019,Zenit,NaN,NaN,1,7,21,5,0
1920,2019-08-10,19:00,Premier League,Matchweek 5,Sat,Away,W,2,0,Dynamo Mosc,...,0,2019,Zenit,NaN,NaN,0,3,19,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,2024-04-28,19:15,Premier League,Matchweek 26,Sun,Away,L,0,1,Dynamo Mosc,...,0,2023,Zenit,NaN,NaN,0,3,19,6,0
26,2024-05-06,18:00,Premier League,Matchweek 27,Mon,Away,D,1,1,Fakel Voronezh,...,0,2023,Zenit,NaN,NaN,0,6,18,0,0
27,2024-05-11,19:00,Premier League,Matchweek 28,Sat,Home,L,0,1,CSKA Moscow,...,0,2023,Zenit,NaN,NaN,1,2,19,5,0
28,2024-05-19,19:00,Premier League,Matchweek 29,Sun,Away,W,5,1,Akhmat Grozny,...,0,2023,Zenit,NaN,NaN,0,0,19,6,1


In [67]:
def rolling_averages(group, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset=new_cols)
    return group

In [71]:
cols = ["gf", "ga", "sh", "sot", "pk", "pkatt"]
new_cols = [f"{c}_rolling" for c in cols]

rolling_averages(group, cols, new_cols)

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,opp_code,hour,day_code,target,gf_rolling,ga_rolling,sh_rolling,sot_rolling,pk_rolling,pkatt_rolling
1919,2019-08-03,21:30,Premier League,Matchweek 4,Sat,Home,D,1,1,Krasnodar,...,7,21,5,0,2.000000,0.333333,15.000000,5.333333,0.666667,0.666667
1920,2019-08-10,19:00,Premier League,Matchweek 5,Sat,Away,W,2,0,Dynamo Mosc,...,3,19,5,1,1.666667,0.333333,16.000000,4.666667,0.666667,0.666667
1921,2019-08-17,19:00,Premier League,Matchweek 6,Sat,Home,D,0,0,Akhmat Grozny,...,0,19,5,0,1.666667,0.333333,14.000000,4.000000,0.333333,0.333333
1922,2019-08-24,18:30 (16:30),Premier League,Matchweek 7,Sat,Away,L,0,1,Ufa,...,19,18,5,0,1.000000,0.333333,16.666667,4.333333,0.000000,0.000000
1923,2019-09-01,19:00,Premier League,Matchweek 8,Sun,Away,W,1,0,Spartak Moscow,...,16,19,6,1,0.666667,0.333333,19.000000,4.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,2024-04-28,19:15,Premier League,Matchweek 26,Sun,Away,L,0,1,Dynamo Mosc,...,3,19,6,0,1.000000,1.000000,14.666667,5.666667,0.000000,0.000000
26,2024-05-06,18:00,Premier League,Matchweek 27,Mon,Away,D,1,1,Fakel Voronezh,...,6,18,0,0,0.333333,1.000000,14.000000,4.666667,0.000000,0.000000
27,2024-05-11,19:00,Premier League,Matchweek 28,Sat,Home,L,0,1,CSKA Moscow,...,2,19,5,0,0.333333,1.333333,14.333333,5.333333,0.000000,0.000000
28,2024-05-19,19:00,Premier League,Matchweek 29,Sun,Away,W,5,1,Akhmat Grozny,...,0,19,6,1,0.333333,1.000000,14.000000,4.333333,0.000000,0.000000


In [74]:
matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols)).reset_index(drop=True)

C:\Users\Smetanin Ilia\AppData\Local\Temp\ipykernel_1040\2439766232.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols)).reset_index(drop=True)


In [75]:
matches_rolling

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,opp_code,hour,day_code,target,gf_rolling,ga_rolling,sh_rolling,sot_rolling,pk_rolling,pkatt_rolling
0,2019-08-05,20:00,Premier League,Matchweek 4,Mon,Home,W,2,1,Orenburg,...,10,20,0,1,0.333333,1.333333,12.000000,4.000000,0.0,0.333333
1,2019-08-11,21:30,Premier League,Matchweek 5,Sun,Home,L,1,3,Spartak Moscow,...,16,21,6,0,0.666667,1.666667,10.000000,2.666667,0.0,0.000000
2,2019-08-17,19:00,Premier League,Matchweek 6,Sat,Away,D,0,0,Zenit,...,21,19,5,0,1.000000,1.666667,8.333333,2.333333,0.0,0.000000
3,2019-08-25,16:30,Premier League,Matchweek 7,Sun,Away,L,0,3,CSKA Moscow,...,2,16,6,0,1.000000,1.333333,7.666667,1.666667,0.0,0.000000
4,2019-08-31,19:00,Premier League,Matchweek 8,Sat,Home,D,1,1,Tambov,...,17,19,5,0,0.333333,2.000000,7.000000,1.333333,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2317,2024-04-28,19:15,Premier League,Matchweek 26,Sun,Away,L,0,1,Dynamo Mosc,...,3,19,6,0,1.000000,1.000000,14.666667,5.666667,0.0,0.000000
2318,2024-05-06,18:00,Premier League,Matchweek 27,Mon,Away,D,1,1,Fakel Voronezh,...,6,18,0,0,0.333333,1.000000,14.000000,4.666667,0.0,0.000000
2319,2024-05-11,19:00,Premier League,Matchweek 28,Sat,Home,L,0,1,CSKA Moscow,...,2,19,5,0,0.333333,1.333333,14.333333,5.333333,0.0,0.000000
2320,2024-05-19,19:00,Premier League,Matchweek 29,Sun,Away,W,5,1,Akhmat Grozny,...,0,19,6,1,0.333333,1.000000,14.000000,4.333333,0.0,0.000000


In [76]:
matches_rolling.index = range(matches_rolling.shape[0])

In [79]:
def make_predictions(data, predictors):
    train = data[data["date"] < '2022-01-01']
    test = data[data["date"] > '2022-01-01']
    RFC.fit(train[predictors], train["target"])
    preds = RFC.predict(test[predictors])
    combined = pd.DataFrame(dict(actual=test["target"], predicted=preds), index=test.index)
    precision = precision_score(test["target"], preds)
    return combined, precision

In [80]:
combined, precision = make_predictions(matches_rolling, predictors + new_cols)

In [82]:
precision

0.4752475247524752

In [83]:
combined = combined.merge(matches_rolling[["date", "team", "opponent", "result"]], left_index=True, right_index=True)

In [84]:
combined

,actual,predicted,date,team,opponent,result
75,1,1,2022-02-27,Akhmat Grozny,Ufa,W
76,0,1,2022-03-07,Akhmat Grozny,Rubin Kazan,L
77,0,0,2022-03-13,Akhmat Grozny,Ural,D
78,0,0,2022-03-19,Akhmat Grozny,Loko Moscow,L
79,0,0,2022-04-02,Akhmat Grozny,Arsenal Tula,D
...,...,...,...,...,...,...
2317,0,0,2024-04-28,Zenit,Dynamo Mosc,L
2318,0,0,2024-05-06,Zenit,Fakel Voronezh,D
2319,0,0,2024-05-11,Zenit,CSKA Moscow,L
2320,1,0,2024-05-19,Zenit,Akhmat Grozny,W


In [ ]:
#class MissingDict(dict):
#    __missing__ = lambda self, key: key

#map_values = {"Brighton and Hove Albion": "Brighton", "Manchester United": "Manchester Utd", "Newcastle United": "Newcastle Utd", "Tottenham Hotspur": "Tottenham", "West Ham United": "West Ham", "Wolverhampton Wanderers": "Wolves"} 
#mapping = MissingDict(**map_values)

In [86]:
merged = combined.merge(combined, left_on=["date", "team"], right_on=["date", "opponent"])

In [87]:
merged

,actual_x,predicted_x,date,team_x,opponent_x,result_x,actual_y,predicted_y,team_y,opponent_y,result_y
0,1,1,2022-02-27,Akhmat Grozny,Ufa,W,0,0,Ufa,Akhmat Grozny,L
1,0,1,2022-03-07,Akhmat Grozny,Rubin Kazan,L,1,0,Rubin Kazan,Akhmat Grozny,W
2,0,0,2022-03-13,Akhmat Grozny,Ural,D,0,1,Ural Yekaterinburg,Akhmat Grozny,D
3,0,0,2022-03-19,Akhmat Grozny,Loko Moscow,L,1,0,Lokomotiv Moscow,Akhmat Grozny,W
4,0,0,2022-04-02,Akhmat Grozny,Arsenal Tula,D,0,0,Arsenal Tula,Akhmat Grozny,D
...,...,...,...,...,...,...,...,...,...,...,...
915,0,0,2024-04-28,Zenit,Dynamo Mosc,L,1,0,Dynamo Moscow,Zenit,W
916,0,0,2024-05-06,Zenit,Fakel Voronezh,D,0,0,Fakel Voronezh,Zenit,D
917,0,0,2024-05-11,Zenit,CSKA Moscow,L,1,0,CSKA Moscow,Zenit,W
918,1,0,2024-05-19,Zenit,Akhmat Grozny,W,0,0,Akhmat Grozny,Zenit,L


In [88]:
merged[(merged["predicted_x"] == 1) & (merged["predicted_y"] ==0)]["actual_x"].value_counts()

actual_x
0    70
1    61
Name: count, dtype: int64

In [89]:
70/131

0.5343511450381679